# log-back — faded example 3: Fill the autograd witness for log_back

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-back`. Running the beacon reports progress on the `Backprop: log_back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: log_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-back"
DD_SUBTOPIC = "Backprop: log_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To validate a hand-rolled backward, build the input with `requires_grad_(True)`, run the forward through `torch.log`, scale by `grad_out`, sum to a scalar, and call `.backward()`. The resulting `.grad` is the ground-truth gradient to compare against.

## Faded exercise 3

Complete `autograd_grad(x, grad_out)` which returns the autograd gradient of `(log(x)*grad_out).sum()` w.r.t. x. Fill in the backward call that populates the gradient.

**Fill in:** the .backward() call on the scalar loss that populates xv.grad

In [ ]:
import torch as t

def autograd_grad(x, grad_out):
    xv = x.clone().detach().requires_grad_(True)
    loss = (t.log(xv) * grad_out).sum()
    raise NotImplementedError()  # TODO: the .backward() call on the scalar loss that populates xv.grad
    return xv.grad


def _test():
    t.manual_seed(13)
    x = t.rand(7) + 0.5
    grad_out = t.randn(7)
    g = autograd_grad(x, grad_out)
    # independent ground truth from the closed form grad_out / x
    assert g is not None
    assert t.allclose(g, grad_out / x, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def autograd_grad(x, grad_out):
    xv = x.clone().detach().requires_grad_(True)
    loss = (t.log(xv) * grad_out).sum()
    loss.backward()
    return xv.grad
```
</details>